## 🔥 <span style='text-decoration: double underline; color:rgb(10,110,217)'>**PyTorch: Training and Data Pipelines**</span>

**Author:** Ángel Fernández Caravaca

**GitHub:** [@afdezcaravaca](https://github.com/afdezcaravaca)

**Date:** March 2026

### 📑  <span style='color:rgb(10,110,217)'><u>**Introduction**</u></span>

#### 🏗️ <span style='color:rgb(10,110,217)'><u>**Building Models with nn.Module**</u></span>

`nn.Module` is the cornerstone of every neural network in PyTorch. Any model — from a single linear layer to a transformer — must subclass it.

The minimal contract requires two things: defining learnable components in `__init__` and describing the forward computation in `forward()`.

```python
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(in_features, out_features)

    def forward(self, x):
        return self.layer(x)
```

When you call `model(x)`, PyTorch routes the call through `forward()` automatically, while also handling gradient hooks, device placement, and parameter tracking behind the scenes.

**Linear Regression** is the simplest possible model: a single `nn.Linear(1, 1)` layer, which learns two parameters — a weight `w` and a bias `b` — to fit `ŷ = wx + b`. The training loop follows a four-step cycle:

1. **Forward pass** — compute predictions
2. **Loss computation** — measure the error (e.g., MSE)
3. **Backward pass** — compute gradients via `loss.backward()`
4. **Parameter update** — apply `optimizer.zero_grad()`, then `optimizer.step()`. 


#### 📈 <span style='color:rgb(10,110,217)'><u>**Activation Functions**</u></span>

Activation functions introduce **non-linearity** into the network, enabling it to learn complex mappings beyond simple linear transformations. Without them, stacking multiple layers would be mathematically equivalent to a single linear layer. Some examples are:

<div align='center'>

| Activation | Formula | Key Properties |
|---|---|---|
| **ReLU** | `max(0, x)` | Simple, fast; suffers from *dying ReLU* for negative inputs |
| **Tanh** | `(eˣ − e⁻ˣ) / (eˣ + e⁻ˣ)` | Zero-centered output `(−1, 1)`; can vanish for large inputs |
| **GELU** | `x · Φ(x)` | Smooth, probabilistic gating; preferred in Transformers |
| **LeakyReLU** | `max(αx, x)` | Fixes dying ReLU by allowing small negative gradients |
</div>


#### 🌘 <span style='color:rgb(10,110,217)'><u>**Decision Boundaries**</u></span>

In classification problems, **decision boundaries** are the regions in feature space where the model's predicted class changes. They represents the limits the model has stablished to distinguish between classes. 

Visualizing them via a meshgrid of predictions reveals how well the MLP has carved the input space into class regions.

<div align='center'>
<img src="https://imgs.search.brave.com/KGUFJAitARuIC-gBXI2EpaNbWzY6uJuLouSxRmJf5sI/rs:fit:860:0:0:0/g:ce/aHR0cHM6Ly9yb2hp/dG1pZGhhMjMuZ2l0/aHViLmlvL2Fzc2V0/cy9pbWFnZXMvTmV1/cmFsTmV0Qm91bmRh/cnkvZGVjaXNpb24t/Ym91bmRhcnkucG5n" alt="Decision boundaries" width="400">

</div>

#### 👮 <span style='color:rgb(10,110,217)'><u>**Inspecting Model Parameters**</u></span>

PyTorch tracks all `nn.Parameter` objects (and the parameters of submodules) automatically. Two key APIs:

- `model.named_parameters()` — iterates over `(name, tensor)` pairs for all learnable parameters
- `model.parameters()` — same, but without names; useful for passing to an optimizer

The total parameter count is:

```python
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
```

Visualizing weight matrices as heatmaps (e.g., with `matplotlib.imshow`) can reveal patterns such as structured sparsity or poor initialization — insights that raw numbers alone do not provide.

#### 💾 <span style='color:rgb(10,110,217)'><u>**Saving and Loading Models**</u></span>

PyTorch separates the **model architecture** (Python code) from the **learned weights** (a `state_dict`, i.e., an ordered dictionary of parameter tensors).

```python
# Save
torch.save(model.state_dict(), 'model.pth')

# Load into a new instance of the same architecture
model2 = MyModel()
model2.load_state_dict(torch.load('model.pth'))
model2.eval()
```

This design is intentional: weights are lightweight and portable; the architecture is defined in code and can be versioned independently. Using `torch.allclose()` to verify that two models produce identical outputs after loading confirms that no numerical drift occurred.

- **Note:** Always use `.pth` to save the model.

#### 🖌️ <span style='color:rgb(10,110,217)'><u>**Custom Layers and Learnable Parameters**</u></span>

Beyond the built-in layers, PyTorch allows you to define entirely new operations that participate in gradient computation. Any tensor wrapped in `nn.Parameter` is:

- Registered as a model parameter (visible via `.parameters()`)
- Included in optimizer updates
- Tracked by autograd

#### ♻️<span style='color:rgb(10,110,217)'><u>**Residual Connections (ResNets)**</u></span>

A **Residual Block** introduces a skip connection that adds the block's input directly to its output:

```
out = F(x) + x
```

where `F(x)` is a sequence of linear layers and activations. This seemingly small change solves two critical problems in deep networks:

**Vanishing gradients**: During backpropagation, gradients must pass through every layer. In deep networks, repeated multiplication by small numbers causes the gradient to shrink exponentially. Skip connections provide an unobstructed gradient highway directly to earlier layers.

**Degradation problem**: Empirically, simply adding more layers to a plain network can *hurt* performance, even on training data. With residual connections, deeper networks can always fall back to the identity function (`F(x) ≈ 0`), guaranteeing that adding depth never makes things worse.

The key insight is that **learning a residual `F(x) = H(x) − x` is easier than learning the full mapping `H(x)` from scratch**, especially when the optimal function is close to the identity.

<div align='center'>
<img src="https://imgs.search.brave.com/GK5ZWgwQdzf4rH-krwOrDokh7barX0iVDRlNZ1BjJ9g/rs:fit:860:0:0:0/g:ce/aHR0cHM6Ly91cGxv/YWQud2lraW1lZGlh/Lm9yZy93aWtpcGVk/aWEvY29tbW9ucy90/aHVtYi9iL2JhL1Jl/c0Jsb2NrLnBuZy8x/MjgwcHgtUmVzQmxv/Y2sucG5n" alt="Decision boundaries" width="400">

</div>


#### 🎊 <span style='color:rgb(10,110,217)'><u>**Weight Initialization**</u></span>

How parameters are initialized before training profoundly affects whether and how fast a network learns. Poor initialization causes two classic failure modes:

- **Symmetry breaking problem (all-zeros)**: If all weights are initialized to zero, every neuron in a layer computes the same function, receives the same gradient, and updates identically — effectively collapsing the layer to a single neuron. The network cannot break this symmetry through gradient descent alone.

There are different ways of initialization weights. For example:

- **Xavier (Glorot) initialization**: Designed for saturating activations like Tanh and Sigmoid. Scales weights so that the **variance of activations and gradients is preserved** across layers:

$$
W \approx \text{Uniform}(−\sqrt{\frac{6}{(n_{in} + n_{out})}}, \sqrt{\frac{6}{(n_{in} + n_{out})}})
$$

The goal is to prevent activations from vanishing (too small) or exploding (too large) as the signal propagates forward and the gradient propagates backward.

- **Kaiming (He) initialization**: Designed specifically for **ReLU-family activations**, which discard roughly half of all activations (the negative half). Xavier's variance calculation assumes a symmetric, linear activation, so it underestimates the effective fan-in for ReLU. Kaiming corrects for this:

$$
W \approx  \text{Normal}(0, \sqrt{\frac{2}{n_{in}}})
$$

The factor of 2 compensates for the halving effect of ReLU.

- <span style='color:rgb(188, 7, 194)'>**Rule of thumb**:</span> Use Xavier for Tanh/Sigmoid; use Kaiming for ReLU, LeakyReLU, and GELU.

---

### 🦉 <span style='color:rgb(10,110,217)'><u>**Level 2: Seeing through the layers**</u></span>
Use the [official documentation](https://docs.pytorch.org/docs/stable/index.html) while solving the exercises.

##### <span style='color:rgb(10,110,217)'>🔢**Exercise 1:**</span>

> Build a simple linear regression model using `nn.Module` from scratch. Define a class `LinearRegression` with one `nn.Linear(1, 1)` layer. Generate synthetic data following `y = 4x + 3 + noise` with 100 points. Train for 400 epochs using `MSELoss` and `SGD`. After training, verify that the learned weight is close to **4** and the bias close to **3**.
- **Hint:** 
    - Architecture: 1 → Linear(1,1) → 1
    - Loss:         `nn.MSELoss`
    - Optimizer:    `torch.optim.SGD`

I won't use here `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)` because the gradients won't explode. The general rule of use is to use it when the loss explodes. That would be an extra-hyperparameter in our model.

##### <span style='color:rgb(10,110,217)'>🔢**Exercise 2:**</span>

> Train the same MLP architecture four times on `y = sin(x)` (300 evenly spaced points, `x ∈ [-π, π]`), changing **only** the activation function each time: `ReLU`, `Tanh`, `GELU` and `LeakyReLU`. The activation is applied after each hidden layer; the output layer has none.
>
>Plot the four loss curves on the same graph and the four learned functions vs. the real sine curve. Which activation fits the sine shape best, and why?

- **Architecture:** `Linear(1→32) → Act → Linear(32→16) → Act → Linear(16→1)`
- **Loss:** `nn.MSELoss`
- **Optimizer:** `Adam (lr=1e-3)`
- **Epochs:** `500`
- **Batch:** full-batch (all 300 points per step)

##### <span style='color:rgb(10,110,217)'>🔢**Exercise 3:**</span>
> Generate a synthetic 3-class dataset in 2D with `sklearn.datasets.make_blobs(300 samples, centers=3, cluster_std=1.2, random_state=42)`. Split it
> 80/20 into train/test (`random_state=42`). Build and train a MLP, then:
>
> - Print the final **accuracy** on the test set
> - Plot the **decision boundaries** as a `plt.contourf` colored background
>   (meshgrid step `0.1`, margin `0.5` beyond data range) with the test points
>   overlaid as a scatter plot

- **Architecture:** `Linear(2→16) → ReLU → Linear(16→8) → ReLU → Linear(8→3)`
- **Loss:** `nn.CrossEntropyLoss` (expects raw logits — no activation on output layer)
- **Optimizer:** `Adam (lr=1e-3)`
- **Epochs:** `300`
- **Batch:** full-batch (all training points per step)
- **Tensors:** `X` → `FloatTensor`, `y` → `LongTensor`

##### <span style='color:rgb(10,110,217)'>🔢**Exercise 4:**</span>

> Define a MLP with `nn.Sequential` and inspect its parameters **without
> using any external library**:
>
> 1. Print the name and shape of every parameter using `model.named_parameters()`
> 2. Count the **total number of trainable parameters** with `sum(p.numel() for p in model.parameters())`
> 3. Visualize the weight matrix of the first layer as a **heatmap** using
>    `plt.imshow` with `cmap='viridis'` and a colorbar

- **Architecture:** `Linear(4→16) → ReLU → Linear(16→8) → ReLU → Linear(8→3)`
- **No training needed** — just instantiate the model with default (random) weights

##### <span style='color:rgb(10,110,217)'>🔢**Exercise 5:**</span>

> Train a MLP classifier on the Iris dataset (`sklearn.datasets.load_iris`,
> 150 samples, 4 features, 3 classes). Once trained:
>
> 1. Save the model weights with `torch.save(model.state_dict(), 'iris_model.pth')`
> 2. Create a **new instance** of the same architecture and load the saved weights
>    with `model.load_state_dict(torch.load('iris_model.pth'))`
> 3. Run both models in `model.eval()` + `torch.no_grad()` on the test set
> 4. Verify the outputs are **identical** with `torch.allclose(out1, out2)`

- **Architecture:** `Linear(4→16) → ReLU → Linear(16→8) → ReLU → Linear(8→3)`
- **Loss:** `nn.CrossEntropyLoss` (expects raw logits — no activation on output layer)
- **Optimizer:** `Adam (lr=1e-3)`
- **Epochs:** `200`
- **Batch:** full-batch (all training points per step)
- **Split:** 80/20 train/test (`random_state=42`)
- **Tensors:** `X` → `FloatTensor`, `y` → `LongTensor`

##### <span style='color:rgb(10,110,217)'>🔢**Exercise 6:**</span>
> Implement a custom layer `ScaledLinear(nn.Module)` that performs
> `y = alpha * (W · x + b)`, where `alpha` is a **learnable scalar**
> initialized to `1.0` via `nn.Parameter`. Use it to build a MLP that
> predicts **both** `y₁ = sin(x)` and `y₂ = cos(x)` simultaneously
> (300 evenly spaced points, `x ∈ [-π, π]`, full-batch). Stack them as
> `Y = torch.stack([y1, y2], dim=1)` of shape `(300, 2)`. Then:
>
> 1. Verify `alpha` appears in every layer when calling `model.named_parameters()`
> 2. Print the value of `alpha` in each `ScaledLinear` before and after training
> 3. Did `alpha` change? Print `True`/`False` per layer comparing initial vs final value
> 4. Plot both predicted curves vs. the real ones in a single figure with a legend

- **Architecture:** `ScaledLinear(1→32) → ReLU → ScaledLinear(32→16) → ReLU → ScaledLinear(16→2)`
- **Loss:** `nn.MSELoss` — applied directly on the full output of shape `(300, 2)`
- **Optimizer:** `Adam (lr=1e-3)` — **all** parameters including `alpha`
- **Epochs:** `1000`
- **Batch:** full-batch (all 300 points per step)
- **Tensors:** `X` → `FloatTensor` of shape `(300, 1)`, `Y` → `FloatTensor` of shape `(300, 2)`


---

### 🎌<span style='color:rgb(10,110,217)'><u>**Summary**</u></span>

<div align='center'>

| Topic | Key Concept | PyTorch API |
|---|---|---|
| Model definition | Subclass `nn.Module`, implement `forward` | `nn.Module`, `nn.Linear` |
| Regression | MSE loss, SGD/Adam | `nn.MSELoss`, `torch.optim` |
| Activation functions | Non-linearity, inductive bias | `nn.ReLU`, `nn.Tanh`, `nn.GELU`, `nn.LeakyReLU` |
| Multi-class | Logits + cross-entropy | `nn.CrossEntropyLoss` |
| Parameter inspection | Count and visualize weights | `.named_parameters()`, `.numel()` |
| Saving / loading | Persist and restore `state_dict` | `torch.save`, `torch.load` |
| Custom layers | Learnable scalars/tensors | `nn.Parameter` |
| Multi-output | Joint regression over multiple targets | Output layer with `k` units |
| Binary classification | Fused sigmoid + BCE for stability | `nn.BCEWithLogitsLoss` |
| Residual blocks | Skip connections for deep networks | Manual `out = F(x) + x` |
| Weight initialization | Break symmetry, control variance | `torch.nn.init` |
</div>